In [1]:
with open("tinyphilosopher.txt", 'r', encoding = 'utf-8') as f:
    text = f.read()
text[:100]

'the ego builds walls, the soul seeks horizons.\nan inflated ego sees only its own reflection.\ntrue wi'

In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

sys.path.insert(0, str(PROJECT_ROOT))

print(PROJECT_ROOT)

d:\Programs\python (+sql)\Learning_Neural_Networks_From_Scratch


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import plotly.graph_objects as go

In [4]:
from my_tokenizer.gpt_bpe_tokenizer.serialization import load_tokenizer
from my_tokenizer.gpt_bpe_tokenizer.regex_tokenizer import RegexTokenizer

In [5]:
tokenizer = load_tokenizer(RegexTokenizer, PROJECT_ROOT / "checkpoints" / "tokenizer" / "gpt4tokenizer.model")

In [6]:
ids = torch.tensor(tokenizer.encode(text))
dataset_size = len(ids)
train_size = int(0.9 * dataset_size)
x_train , x_test= ids[:train_size], ids[train_size:]
y_train, y_test = ids[1:train_size+1], ids[train_size+1:]

In [7]:
T = 8
seq_len = (len(x_train) // T) * T
seq_len2 = (len(x_test) // T) * T
x_train = x_train[:seq_len].view(-1, T)
y_train = y_train[:seq_len].view(-1, T)
x_test = x_test[:seq_len2].view(-1, T)
y_test = y_test[:seq_len2].view(-1, T)

In [8]:
x_train[:10], y_train[:10]

(tensor([[ 116,  263,  519,  103,  111,  360,  117,  465],
         [ 100,  115,  340,  591,  115,   44,  268,  306],
         [ 309,  108,  306,  608,  115,  331,  258,  638],
         [ 262,  115,  294,  266,  332,  102,  108,  551],
         [ 519,  103,  111,  306,  101,  289,  299,  352],
         [ 941,  316, 1018,  432,  102,  108,  477,  338],
         [ 294,  116,  114,  117,  101,  424,  115,  100],
         [ 288,  360,  702,  784,  523,  601,  268,  519],
         [ 103,  111,   32,  550,  115,  294,  116,  263],
         [ 519,  103,  111,  349,  108,  295,  115,  348]]),
 tensor([[ 263,  519,  103,  111,  360,  117,  465,  100],
         [ 115,  340,  591,  115,   44,  268,  306,  309],
         [ 108,  306,  608,  115,  331,  258,  638,  262],
         [ 115,  294,  266,  332,  102,  108,  551,  519],
         [ 103,  111,  306,  101,  289,  299,  352,  941],
         [ 316, 1018,  432,  102,  108,  477,  338,  294],
         [ 116,  114,  117,  101,  424,  115,  100,  2

In [9]:
x_train.shape, y_train.shape

(torch.Size([142484, 8]), torch.Size([142484, 8]))

In [10]:
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [19]:
class LSTMLanguageModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, device = device)
        self.gates = nn.Linear(embedding_dim + hidden_size, 4 * hidden_size, device = device)
        self.lm_head = nn.Linear(hidden_size, vocab_size, device = device)
        self.hidden_size = hidden_size
    def forward(self, x, h_prev=None, c_prev=None):
        emb = self.embedding(x)
        B, T = x.shape
        
        if h_prev is None:
            h_prev = torch.zeros(B, self.hidden_size, device = x.device)
        if c_prev is None:
            c_prev = torch.zeros(B, self.hidden_size, device = x.device)
        
        logits_list = []
        for t in range(T):
            x_t = emb[:, t, :]

            combined = torch.cat([x_t, h_prev], dim=1)
            gates = self.gates(combined)

            f_t, i_t, c_hat_t, o_t = gates.chunk(4, dim=1)

            f_t = torch.sigmoid(f_t)
            i_t = torch.sigmoid(i_t)
            c_hat_t = torch.tanh(c_hat_t)
            o_t = torch.sigmoid(o_t)

            c_t = f_t * c_prev + i_t * c_hat_t
            h_t = o_t * torch.tanh(c_t)

            logits = self.lm_head(h_t)
            logits_list.append(logits)

            h_prev = h_t
            c_prev = c_t
        
        logits = torch.stack(logits_list, dim = 1)
        return logits, (h_t, c_t)
    

In [26]:
vocab_size = 1024
embedding_dim = 42
hidden_size = 150
batch_size = 32
lr = 0.001
steps = 5000

model = LSTMLanguageModel(vocab_size, embedding_dim, hidden_size)
optimizer = optim.AdamW(model.parameters(), lr=lr)

In [27]:
def get_loss(split):

    if split == "train":
        model.train()
        x_dataset, y_dataset = x_train, y_train
    else:
        model.eval()
        x_dataset, y_dataset = x_test, y_test
    
    with torch.no_grad():

        row, _ = x_dataset.shape
        
        idx = torch.randint(0, row, (batch_size,))
        
        x_batch = x_dataset[idx]
        y_batch = y_dataset[idx]
        
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        B, T = x_batch.shape
        
        logits, _ = model(x_batch)

        loss = F.cross_entropy(logits.reshape(B*T, vocab_size), y_batch.reshape(B*T))
        
        return loss.item()

train_loss = get_loss("train")
val_loss = get_loss("test")
train_loss, val_loss

(6.930760860443115, 6.934903621673584)

In [28]:
train_losses = []
val_losses = []
for i in range(steps):

    model.train() #For batch norm (if used)
    #Forward pass
    #sample batch
    row, _ = x_train.shape
    idx = torch.randint(0, row, (batch_size,))

    x_batch = x_train[idx]
    y_batch = y_train[idx]

    x_batch = x_batch.to(device)
    y_batch = y_batch.to(device)

    B, T = x_batch.shape

    logits, _ = model(x_batch)
    loss = F.cross_entropy(logits.reshape(B*T, vocab_size), y_batch.reshape(B*T))

    #Backward pass
    optimizer.zero_grad()
    loss.backward()

    #Gradient Clipping
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    
    #Update
    optimizer.step()

    #Print loss
    if i%100 == 0:
        train_loss = get_loss("train")
        val_loss = get_loss("test")
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        print(f"Step:{i}/{steps} train loss:{train_loss:.4f} test_loss:{val_loss:.4f}")


Step:0/5000 train loss:6.9243 test_loss:6.9263
Step:100/5000 train loss:5.4243 test_loss:5.3249
Step:200/5000 train loss:5.0028 test_loss:4.9425
Step:300/5000 train loss:4.7982 test_loss:4.8428
Step:400/5000 train loss:4.5896 test_loss:4.6228
Step:500/5000 train loss:4.5385 test_loss:4.5355
Step:600/5000 train loss:4.1880 test_loss:4.3371
Step:700/5000 train loss:4.0286 test_loss:4.0166
Step:800/5000 train loss:4.0201 test_loss:3.5798
Step:900/5000 train loss:3.7955 test_loss:3.8295
Step:1000/5000 train loss:3.8006 test_loss:3.8526
Step:1100/5000 train loss:3.5425 test_loss:3.7342
Step:1200/5000 train loss:3.7792 test_loss:3.5610
Step:1300/5000 train loss:3.4076 test_loss:3.3721
Step:1400/5000 train loss:3.5522 test_loss:3.4262
Step:1500/5000 train loss:3.5196 test_loss:3.4937
Step:1600/5000 train loss:3.1407 test_loss:3.3689
Step:1700/5000 train loss:3.3796 test_loss:3.1805
Step:1800/5000 train loss:3.3640 test_loss:3.0856
Step:1900/5000 train loss:3.1222 test_loss:3.1796
Step:2000/50

In [29]:
steps = torch.arange(1, len(train_losses)) 
fig = go.Figure()

fig.add_trace(go.Scatter(x=steps, y=train_losses, mode='lines', name='Training Loss', line=dict(width=2)))

fig.add_trace(go.Scatter(x=steps, y=val_losses, mode='lines', name='Validation Loss',line=dict(width=2, dash='dash')))

fig.update_layout(title='LSTM Training and Validation Loss',xaxis_title='Steps', yaxis_title='Loss', template='plotly_white', hovermode='x unified')

fig.show()

In [31]:
loss.item(), train_loss, val_loss

(2.7763566970825195, 3.1014671325683594, 2.9605977535247803)

In [35]:
#Sampling from the model
model.eval()
with torch.no_grad():
    max_tokens = 300
    token = torch.zeros((1, 1), dtype=torch.long, device=device)
    
    h_prev = None
    c_prev = None

    generated = []

    for _ in range(max_tokens):
        logits, (h_prev, c_prev) = model(token, h_prev, c_prev)
        logits = logits[:, -1, :]
        probs = F.softmax(logits, dim=-1)
        token = torch.multinomial(probs, num_samples= 1)
        generated.append(token.item())

    output = tokenizer.decode(generated)
    print(output)

f claims the imfferritory reminit authenticic drauma.
itrelcere the awaits in no truly sanction in vecriendship.
then its true the unwritten loes grace.
true sreving fill't is poering see theirness, ad landscape a mystistic their trace.
does the shat diversal.
every moments.
the memplabor.
even home.
every reshers gift.
lass intive.
the common outlaour spiritpimes without ruin.
thred only essentity.
to tany on stsil of life is to accept leaving experience.
our thoughts an illusion is to heerail, ignents.
the body's louto an a decfferstensy, potion spoken the whole, the sorrant be a many dimensions.
somel reflect an the final
